# Week 1 · Day 4 — Lab 2
## Exploratory Analysis in Jupyter

> **AI Engineering Academy** · Gamut Technology Services

Jupyter is where you *understand* a dataset before building anything on it. Used
well, it's the fastest path from raw data to insight. Used carelessly, it's a
reproducibility minefield — the **hidden-state problem**, stale outputs, and
out-of-order execution silently corrupt results. This lab does both halves: the
disciplined **exploratory workflow** (inspect → validate → quality-check →
visualize) and the **habits** that keep a notebook trustworthy.

### ⚙️ Local API — no internet required
The setup cell starts the same local FastAPI server (`lab_api.py`) on
`http://127.0.0.1:8002`, fetches all 252 events, and loads them into a DataFrame
`df` for you. From here on, the work is pandas + matplotlib exploration.

### Learning objectives
1. Run the standard **inspection sequence**: `shape`, `dtypes`, `isnull().sum()`, `describe`, `sample`, `value_counts`, `duplicated`.
2. Use Jupyter **magics** (`%%time`, `%timeit`) to profile, and understand the kernel/namespace model.
3. Write a **`validate_schema`** guard that asserts expected columns and dtypes.
4. Produce a **data-quality report** and flag columns over a null threshold.
5. Create quick **visualizations** (histogram, bar, correlation heatmap, scatter).
6. Explain and defend against the **hidden-state problem** with "Restart & Run All".

### Time budget — ~88 min
| Segment | Time |
|---|---|
| Setup & fetch | 6 min |
| **A.** Inspection patterns | 14 min |
| **B.** Magics & the kernel model | 12 min |
| **C.** Schema validation | 14 min |
| **D.** Data-quality report | 14 min |
| **E.** Visualization | 14 min |
| **F.** Hidden state & reproducibility | 12 min |
| Wrap-up + stretch | 2 min |

### Files you need (beside this notebook)
- `lab_api.py` — the local practice API (started for you).
- `API_REFERENCE.md` — endpoint documentation.


In [ ]:
%pip install --upgrade matplotlib

In [ ]:
# --- Setup: start the local API, fetch events, build a DataFrame -----------
from pathlib import Path
import os, time, requests, sys
import numpy as np
import pandas as pd

# The following code is only required when running from the SOLUTION

# Jupyter notebooks do not define __file__; search upward from the current
# working directory until we find the folder that contains lab_api.py.
lab_api_dir = None
for root in (Path.cwd(), *Path.cwd().parents):
    matches = list(root.glob('**/lab_api.py'))
    if matches:
        lab_api_dir = matches[0].parent
        break
if lab_api_dir is None:
    raise FileNotFoundError('Could not locate lab_api.py from the current notebook directory')
sys.path.append(str(lab_api_dir))

from lab_api import start_server

os.environ.setdefault("LAB_API_KEY", "local-dev-key")
os.environ.setdefault("API_KEY", os.environ["LAB_API_KEY"])
BASE_URL = "http://127.0.0.1:8002"

server, _thread = start_server(port=8002)
for _ in range(50):
    try:
        if requests.get(f"{BASE_URL}/health", timeout=1).status_code == 200:
            break
    except requests.exceptions.RequestException:
        time.sleep(0.1)

def auth_headers():
    return {"Authorization": f"Bearer {os.environ['API_KEY']}", "Accept": "application/json"}

def fetch_all_events(per_page=100):
    """Provided for you (this is the Lab 1 pattern) — returns all event dicts."""
    records, page = [], 1
    with requests.Session() as s:
        s.headers.update(auth_headers())
        while True:
            r = s.get(f"{BASE_URL}/v1/events", params={"page": page, "per_page": per_page}, timeout=(5, 30))
            r.raise_for_status()
            batch = r.json()["data"]
            records.extend(batch)
            if len(batch) < per_page:
                break
            page += 1
    return records

records = fetch_all_events()
df = pd.DataFrame(records)
df["created_at"] = pd.to_datetime(df["created_at"])   # parse the ISO strings to datetime
print("Fetched", len(df), "rows into a DataFrame")

def check(label, predicate):
    try:
        ok = bool(predicate())
    except Exception as exc:
        ok = False
        label = f"{label}  (raised {type(exc).__name__}: {exc})"
    print(("PASS " if ok else "FAIL "), label)
    return ok

print("ready.")

## Part A — The inspection sequence  *(guided)*

Run these in the first cells of every exploratory notebook. They catch schema
surprises before you build on bad assumptions.


In [ ]:
print("shape:", df.shape)
print(df.dtypes)
print("\nnulls per column:")
print(df.isnull().sum())
df.sample(3, random_state=0)

### Exercise A1 — Profile the fetched data
Compute: `n_rows`, `n_cols` (from `df.shape`); `category_nulls` (nulls in the
`category` column); `dup_count` (number of duplicate rows via `df.duplicated().sum()`);
and `top_model` (the most common value in `model`).


In [ ]:
n_rows, n_cols = df.shape
category_nulls = int(df["category"].isnull().sum())
dup_count = int(df.duplicated().sum())
top_model = df["model"].value_counts().idxmax()
print(f"rows={n_rows} cols={n_cols} | category nulls={category_nulls} | dups={dup_count} | top model={top_model}")

In [ ]:
check("A1: 252 rows, 11 columns", lambda: n_rows == 252 and n_cols == 11)
check("A1: category has 20 nulls", lambda: category_nulls == 20)
check("A1: found 2 duplicate rows", lambda: dup_count == 2)
check("A1: top model is atlas-pro", lambda: top_model == "atlas-pro")

🧑‍🏫 **Instructor note — A1.** The two "surprises" seeded into this data are the
point: **20 null categories** (~8%) and **2 duplicate rows**. Both are invisible
until you run `isnull().sum()` and `duplicated().sum()` — which is exactly why those
belong in the first cells. Note strings report the pandas 3.x `str` dtype and
`created_at` is a real `datetime64[us]` because the preamble parsed it.


## Part B — Magics & the kernel model

The kernel is a live Python process; every cell shares one namespace. Magics profile
and introspect that process. `%%time` times a whole cell; `%timeit` times one line
over many runs (great for comparing implementations).


In [ ]:
%%time
# time a whole cell — here, re-fetching from the API
_ = fetch_all_events()

In [ ]:
# %timeit compares implementations. Vectorized vs apply for the same result:
%timeit df["output_tokens"] * 2
%timeit df["output_tokens"].apply(lambda x: x * 2)

### Exercise B1 — Vectorized equals apply (but faster)
Compute a `chars_per_token` column two ways and confirm they're identical:
`cpt_vectorized` = `df["response_length"] / df["output_tokens"]` (vectorized), and
`cpt_apply` = the same via `df.apply(lambda r: r["response_length"] / r["output_tokens"], axis=1)`.
Set `same` to whether they're equal (use `.equals` after aligning names, or compare
numpy arrays).


In [ ]:
cpt_vectorized = df["response_length"] / df["output_tokens"]
cpt_apply = df.apply(lambda r: r["response_length"] / r["output_tokens"], axis=1)
same = bool(np.allclose(cpt_vectorized.to_numpy(), cpt_apply.to_numpy()))
print("identical result:", same)

In [ ]:
check("B1: vectorized and apply give the same numbers", lambda: same is True)
check("B1: result has one value per row", lambda: len(cpt_vectorized) == len(df))

🧑‍🏫 **Instructor note — B.** `%%time` (wall + CPU for a cell) vs `%timeit` (mean ±
std over many runs of one line) — use `%timeit` to *compare*, `%%time` to spot-check.
B1 reinforces Day 3's lesson with a profiler: same answer, very different speed.
Mention `%whos` to audit what's loaded in the kernel and `%load_ext autoreload` for
Lab 3's module workflow.


## Part C — Schema validation

Before cleaning or analysis, assert the data is shaped the way you expect. A schema
guard catches upstream changes (a renamed column, an int that became a float when
nulls appeared) at the *source*, not three steps downstream.


### Exercise C1 — Write `validate_schema`
Implement `validate_schema(df, required)` where `required` maps column name →
expected dtype string. It should raise `AssertionError` if any required column is
missing or has the wrong dtype, and print `"Schema OK"` otherwise. Run it on `df`
with the `REQUIRED` dict below (it should pass), and store the pass result in
`schema_ok`.


In [ ]:
REQUIRED = {
    "event_id": "int64",
    "model": "str",
    "category": "str",
    "score": "float64",
    "created_at": "datetime64[us]",
}

def validate_schema(df, required):
    missing = set(required) - set(df.columns)
    assert not missing, f"Missing columns: {missing}"
    for col, expected in required.items():
        actual = str(df[col].dtype)
        assert actual == expected, f"Column '{col}': expected {expected}, got {actual}"
    print("Schema OK")
    return True

schema_ok = validate_schema(df, REQUIRED)

In [ ]:
check("C1: schema validation passes on df", lambda: schema_ok is True)

### Exercise C2 — Catch a broken schema
Prove the guard actually fires. Make `broken = df.copy()` and change `score` to a
string dtype (`broken["score"] = broken["score"].astype("str")`). Call
`validate_schema(broken, REQUIRED)` inside a `try/except AssertionError` and set
`caught` to `True` when it raises.


In [ ]:
broken = df.copy()
broken["score"] = broken["score"].astype("str")
caught = False
try:
    validate_schema(broken, REQUIRED)
except AssertionError as exc:
    caught = True
    print("Caught as expected:", exc)

In [ ]:
check("C2: validate_schema raised on the wrong dtype", lambda: caught is True)

🧑‍🏫 **Instructor note — C.** A schema guard is "unit testing for your data." C1
passes (the preamble parsed `created_at`, so it's `datetime64[us]`); C2 proves the
guard has teeth. In Week 2 this same idea becomes a Pandera/Great Expectations
contract — today it's five lines of `assert`. Run it *early*, before any cleaning.


## Part D — Data-quality report

A per-column summary — dtype, null count, null %, distinct values, a sample value —
turns "looks fine" into evidence. Flag columns whose null rate crosses a threshold so
problems announce themselves.


### Exercise D1 — Build the report and flag high-null columns
Implement `data_quality_report(df)` returning a DataFrame indexed by column with
columns `dtype`, `null_count`, `null_pct` (percent, rounded to 2), and `unique_count`.
Build `report`, then set `high_null` to the subset of rows where `null_pct > 5`.
`category` should be the flagged column.


In [ ]:
def data_quality_report(df):
    return pd.DataFrame({
        "dtype": df.dtypes.astype("str"),
        "null_count": df.isnull().sum(),
        "null_pct": (df.isnull().mean() * 100).round(2),
        "unique_count": df.nunique(),
    })

report = data_quality_report(df)
high_null = report[report["null_pct"] > 5]
print(report)
print("\nColumns over 5% nulls:")
print(high_null[["null_pct"]])

In [ ]:
check("D1: report has one row per column", lambda: len(report) == df.shape[1])
check("D1: report columns are correct",
      lambda: set(report.columns) == {"dtype", "null_count", "null_pct", "unique_count"})
check("D1: category is flagged as high-null", lambda: "category" in high_null.index)
check("D1: category null_pct is ~7.9", lambda: abs(report.loc["category", "null_pct"] - 7.94) < 0.5)

🧑‍🏫 **Instructor note — D1.** This report is the deliverable that goes in a markdown
cell titled "Data Quality." The `null_pct > 5` flag surfaces `category` (7.94%)
automatically — the analyst doesn't have to eyeball 11 columns. Point out
`isnull().mean()` gives the *fraction* directly (mean of a boolean), a tidy idiom.


## Part E — Quick visualization

Plots reveal shape that tables hide: distributions, correlations, outliers. Keep them
fast and disposable during exploration.


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

df["score"].hist(bins=30, figsize=(7, 3))
plt.title("Score Distribution"); plt.xlabel("score"); plt.tight_layout()
plt.show()

### Exercise E1 — Correlation structure
Build `numeric_cols = ["score", "input_tokens", "output_tokens", "latency_ms",
"response_length"]`. Compute the correlation matrix `corr`. Then find the pair of
**different** columns with the highest correlation and store their names in
`top_pair` (a set of two names) and the value in `top_corr`. Also draw a scatter of
`output_tokens` vs `response_length` (they should be strongly related by
construction).


In [ ]:
numeric_cols = ["score", "input_tokens", "output_tokens", "latency_ms", "response_length"]
corr = df[numeric_cols].corr()

pairs = corr.unstack()
pairs = pairs[pairs < 0.9999]            # drop the self-correlations on the diagonal
best = pairs.idxmax()
top_pair = set(best)
top_corr = float(pairs.max())
print("strongest pair:", top_pair, "r =", round(top_corr, 3))

df.plot.scatter(x="output_tokens", y="response_length", alpha=0.3, figsize=(6, 4))
plt.title("output_tokens vs response_length"); plt.tight_layout(); plt.show()

In [ ]:
check("E1: corr is a 5x5 matrix", lambda: corr.shape == (5, 5))
check("E1: strongest pair is output_tokens & response_length",
      lambda: top_pair == {"output_tokens", "response_length"})
check("E1: that correlation is strong (> 0.7)", lambda: top_corr > 0.7)

🧑‍🏫 **Instructor note — E1.** `response_length` is `output_tokens × (3–5)`, so the
two correlate strongly — the scatter shows a fan of lines and the corr matrix
confirms it numerically. The `unstack().idxmax()` trick for finding the strongest
off-diagonal pair is worth showing. Encourage a `sns.heatmap(corr, annot=True)` if
seaborn is installed. Plots are for understanding here — don't polish them.


## Part F — Hidden state & reproducibility

The most dangerous notebook bug: a variable computed from an **old** cell, still
sitting in the kernel after you changed the code that produced it. The notebook
*displays* the latest output of each cell, not what a clean top-to-bottom run would
produce. The only defense is **Restart & Run All**.


In [ ]:
# A staged re-creation of the trap, in one cell so it is reproducible:
raw = pd.Series([1, 2, 3, 4], name="raw")

scored = raw * 10            # first definition
mean_score = scored.mean()   # computed now -> 25.0
print("mean_score (from x10):", mean_score)

scored = raw * 100           # you later CHANGE the transform...
# ...but mean_score is NOT recomputed. It is now STALE.
print("mean_score is still:", mean_score, "  <- stale! (kernel holds the old value)")

### Exercise F1 — Recompute to break the staleness
Given the situation above (`scored` is now `raw * 100` but `mean_score` still holds
the old `25.0`), compute `fresh_mean` from the **current** `scored`. Set `is_stale`
to whether the old `mean_score` differs from `fresh_mean`. This is what "Restart &
Run All" guarantees automatically: every value reflects the final code, in order.


In [ ]:
fresh_mean = scored.mean()
is_stale = bool(mean_score != fresh_mean)
print("stale value:", mean_score, "| fresh value:", fresh_mean, "| was stale:", is_stale)

In [ ]:
check("F1: fresh_mean reflects the current transform (250.0)", lambda: fresh_mean == 250.0)
check("F1: the old value was indeed stale", lambda: is_stale is True)

🧑‍🏫 **Instructor note — F.** Make this visceral: `mean_score` shows `25.0` in the
kernel even though `scored` is now `×100`. In a real notebook the stale value can sit
in a chart or a "Findings" cell and get shipped. The fix is a *ritual*: **Restart &
Run All** before trusting, committing, or concluding anything. Point at the execution
brackets `In [n]` — non-contiguous numbers are the warning sign. (Lab 3 takes the
next step: move stable logic into a module you can actually unit-test.)


## Stretch goals *(for fast finishers)*

**S1 — De-duplicate.** Drop the exact duplicate rows into `deduped` with
`df.drop_duplicates()`. It should have 250 rows (252 − 2).

**S2 — Nested → tidy.** Fetch `/v1/articles/nested` (page 1, per_page 20), flatten
with `pd.json_normalize(data, sep="_")`, then `.explode("tags")` into `tags_long` so
each tag is its own row. Confirm it has more rows than the 20 source articles.


In [ ]:
deduped = df.drop_duplicates()
print("after de-dup:", len(deduped))

r = requests.get(f"{BASE_URL}/v1/articles/nested", params={"page": 1, "per_page": 20},
                 headers=auth_headers(), timeout=10)
arts = pd.json_normalize(r.json()["data"], sep="_")
tags_long = arts.explode("tags").reset_index(drop=True)
print("articles:", len(arts), "-> tag rows:", len(tags_long))

In [ ]:
check("S1: de-dup leaves 250 rows", lambda: len(deduped) == 250)
check("S2: exploding tags increased the row count", lambda: len(tags_long) > 20)
check("S2: author fields were flattened", lambda: {"author_name", "author_id"} <= set(arts.columns))

## Wrap-up — what you can now do

- Run the inspection sequence and spot seeded surprises (nulls, duplicates) immediately.
- Profile with `%%time` / `%timeit` and reason about the shared-kernel namespace.
- Guard data with a `validate_schema` that asserts columns and dtypes.
- Produce a data-quality report and auto-flag high-null columns.
- Draw quick distributions, correlations, and scatters.
- Explain the hidden-state problem and defend against it with "Restart & Run All".

**Next:** Lab 3 — recognize when exploratory logic has stabilized, extract it into a
tested importable module, and hand clean data off as Parquet.


In [ ]:
server.should_exit = True
time.sleep(0.3)
print("Local API stopped.")